# Qwen3-TTS Backbone -- `text -> audio_tokens` (verified from real source)

This version is built directly from the actual source of `generate_custom_voice` on your installed
package (you pasted it), not from guessing. The three real stages it uses, copied exactly:

1. Text tokenization: `model._tokenize_texts([model._build_assistant_text(t) for t in texts])`
2. Backbone (talker + code-predictor): `model.model.generate(input_ids=..., instruct_ids=..., languages=..., speakers=...)`
   -> returns `talker_codes_list` -- **this is the audio tokens output you asked for.**
3. Decoder/vocoder: `model.model.speech_tokenizer.decode([{"audio_codes": c} for c in talker_codes_list])`
   -- note this is `model.model.speech_tokenizer`, NOT a separately-loaded `Qwen3TTSTokenizer` instance.
   That mismatch is exactly why the earlier interception attempt captured zero calls.

Same honesty note as before: I can't execute this myself (no GPU/network here), but every line below is
copied or directly derived from the real source you provided, not assumed.


## 0. Setup

In [1]:
import subprocess, sys

def pip_install(args, label):
    proc = subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + args,
                           capture_output=True, text=True)
    print(f"{label}: {'OK' if proc.returncode == 0 else 'FAILED (continuing)'}")
    if proc.returncode != 0:
        print(proc.stderr[-1500:])

pip_install(["-U", "qwen-tts"], "qwen-tts")
pip_install(["soundfile"], "soundfile")

import torch
assert torch.cuda.is_available(), "Select Runtime > Change runtime type > GPU (T4) first."
print("GPU:", torch.cuda.get_device_name(0))


qwen-tts: OK
soundfile: OK
GPU: Tesla T4


## 1. Load the model

In [2]:
import time
from qwen_tts import Qwen3TTSModel

MODEL_ID = "Qwen/Qwen3-TTS-12Hz-0.6B-CustomVoice"
device = "cuda:0"

t0 = time.time()
model = Qwen3TTSModel.from_pretrained(MODEL_ID, device_map=device, dtype=torch.bfloat16)
print(f"Model loaded in {time.time()-t0:.1f}s")

TEST_TEXT = "This is a test sentence for backbone token extraction."
TEST_SPEAKER = model.get_supported_speakers()[0]
TEST_LANGUAGE = "Auto"
print("Using speaker:", TEST_SPEAKER)



    If you do not have SoX, proceed here:
     - - - http://sox.sourceforge.net/ - - -

    If you do (or think that you should) have SoX, double-check your
    path variables.
    



********
********
 


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.81G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/245 [00:00<?, ?B/s]

... (6 outputs left out)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

Model loaded in 24.7s
Using speaker: aiden


In [7]:
import torch

# Your raw tokens, exactly as given
raw_tokens = [151644,  77091,    198,  14990,    678,    600,   1079,  15588,   3611, 71, 151645,    198, 151644,  77091,    198]
raw_input_ids = [torch.tensor([raw_tokens], dtype=torch.int64, device="cuda:0")]

TEST_SPEAKER = model.get_supported_speakers()[0]

# --- Attempt 1: raw tokens, no template wrapping ---
print("=== Raw tokens, unwrapped ===")
try:
    audio_tokens_raw, _ = model.model.generate(
        input_ids=raw_input_ids,
        instruct_ids=[None],
        languages=["Auto"],
        speakers=[TEST_SPEAKER],
        non_streaming_mode=True,
    )
    print("Succeeded. Audio tokens shape:", audio_tokens_raw[0].shape)
except Exception as e:
    print("Failed:", e)

# --- Attempt 2: same tokens, but wrapped in the real template the model expects ---
print("\n=== Same tokens, wrapped in the official template ===")
# Decode your raw ids back to text first, to feed through the real pipeline (requires a text decode step)
decoded_text = model.tokenizer.decode(raw_tokens) if hasattr(model, "tokenizer") else None
print("Decoded text from your raw tokens:", decoded_text)

if decoded_text:
    assistant_text = model._build_assistant_text(decoded_text)
    wrapped_input_ids = model._tokenize_texts([assistant_text])
    audio_tokens_wrapped, _ = model.model.generate(
        input_ids=wrapped_input_ids,
        instruct_ids=[None],
        languages=["Auto"],
        speakers=[TEST_SPEAKER],
        non_streaming_mode=True,
    )
    print("Succeeded. Audio tokens shape:", audio_tokens_wrapped[0].shape)

Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


=== Raw tokens, unwrapped ===
Succeeded. Audio tokens shape: torch.Size([40, 16])

=== Same tokens, wrapped in the official template ===
Decoded text from your raw tokens: None


In [10]:
for name in dir(model):
    if "token" in name.lower() and not name.startswith("_"):
        attr = getattr(model, name, None)
        print(name, "->", type(attr))

In [11]:
assistant_text = model._build_assistant_text("hello all i am hiitesh")
real_input_ids = model._tokenize_texts([assistant_text])

print("Correct Qwen token ids (with template):")
print(real_input_ids[0])
print("shape:", real_input_ids[0].shape)

Correct Qwen token ids (with template):
tensor([[151644,  77091,    198,  14990,    678,    600,   1079,  15588,   3611,
             71, 151645,    198, 151644,  77091,    198]], device='cuda:0')
shape: torch.Size([1, 15])


In [12]:
TEST_SPEAKER = model.get_supported_speakers()[0]

audio_tokens_correct, _ = model.model.generate(
    input_ids=real_input_ids,
    instruct_ids=[None],
    languages=["Auto"],
    speakers=[TEST_SPEAKER],
    non_streaming_mode=True,
)
print("Audio tokens shape:", audio_tokens_correct[0].shape)

wavs, fs = model.model.speech_tokenizer.decode([{"audio_codes": audio_tokens_correct[0]}])
import soundfile as sf
from IPython.display import Audio, display
sf.write("correct_tokens_output.wav", wavs[0], fs)
display(Audio("correct_tokens_output.wav"))

Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


Audio tokens shape: torch.Size([36, 16])


In [13]:
wavs, fs = model.model.speech_tokenizer.decode([{"audio_codes": audio_tokens[0]}])

## 2. The real `backbone()` function

This replicates the text-tokenization + `model.model.generate(...)` call exactly as it appears in
`generate_custom_voice`'s own source, then stops -- it does NOT call `speech_tokenizer.decode(...)`.
That decode step is what you hand off to your decoder/vocoder teammates.


In [3]:
def backbone(text, speaker, language="Auto", instruct=None):
    # Mirrors generate_custom_voice's own logic up to (but not including) the decode step.
    # Input:  text (str), speaker (str, must be in model.get_supported_speakers()), language (str)
    # Output: talker_codes_list -- a list of tensors, the real intermediate audio-token representation
    #         that model.model.speech_tokenizer.decode(...) expects as input.
    texts = [text]
    languages = [language]
    speakers = [speaker]
    instructs = [instruct if instruct else ""]

    if model.model.tts_model_size in "0b6":  # 0.6B model does not support instruct, per the original source
        instructs = [""]

    input_ids = model._tokenize_texts([model._build_assistant_text(t) for t in texts])

    instruct_ids = []
    for ins in instructs:
        if ins is None or ins == "":
            instruct_ids.append(None)
        else:
            instruct_ids.append(model._tokenize_texts([model._build_instruct_text(ins)])[0])

    talker_codes_list, _ = model.model.generate(
        input_ids=input_ids,
        instruct_ids=instruct_ids,
        languages=languages,
        speakers=speakers,
        non_streaming_mode=True,
    )
    return talker_codes_list


t0 = time.time()
audio_tokens = backbone(TEST_TEXT, speaker=TEST_SPEAKER, language=TEST_LANGUAGE)
print(f"backbone() ran in {time.time()-t0:.2f}s")
print("Type:", type(audio_tokens))
print(f"List of {len(audio_tokens)} item(s)")
for i, t in enumerate(audio_tokens):
    print(f"  item[{i}]: type={type(t).__name__}", getattr(t, "shape", ""), getattr(t, "dtype", ""))


Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


backbone() ran in 12.47s
Type: <class 'list'>
List of 1 item(s)
  item[0]: type=Tensor torch.Size([50, 16]) torch.int64


## 3. Verify: decode the captured tokens through the real decoder and listen

Uses `model.model.speech_tokenizer.decode(...)` -- the real decoder object identified from your source,
in the exact input format (`[{"audio_codes": c} for c in talker_codes_list]`) the source uses.


In [4]:
import soundfile as sf
from IPython.display import Audio, display

t0 = time.time()
wavs, fs = model.model.speech_tokenizer.decode([{"audio_codes": c} for c in audio_tokens])
print(f"Decode took {time.time()-t0:.2f}s")

sf.write("backbone_output.wav", wavs[0], fs)
display(Audio("backbone_output.wav"))
print("If this sounds like normal speech, the backbone extraction is confirmed correct.")


Decode took 0.31s


If this sounds like normal speech, the backbone extraction is confirmed correct.


## 4. Cross-check against the official one-shot call

Same text/speaker/language through the normal `generate_custom_voice` -- should sound equivalent to
Section 3's output, since it's running the identical internal steps.


In [5]:
wavs_ref, fs_ref = model.generate_custom_voice(text=TEST_TEXT, speaker=TEST_SPEAKER, language=TEST_LANGUAGE)
sf.write("official_pipeline_output.wav", wavs_ref[0], fs_ref)
display(Audio("official_pipeline_output.wav"))
print("Compare this to backbone_output.wav above.")


Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


Compare this to backbone_output.wav above.


## 5. What to hand your teammates

- **Your deliverable (backbone):** the `backbone(text, speaker, language, instruct=None)` function in
  Section 2. Input: text + speaker + language. Output: `audio_tokens` (a list of tensors -- print the
  shapes from Section 2 and share them with your team so they know the exact format to expect).
- **Decoder/vocoder teammates' input:** whatever `audio_tokens` turned out to be, wrapped as
  `[{"audio_codes": c} for c in audio_tokens]`, fed into `model.model.speech_tokenizer.decode(...)`.
  Share this exact call with them -- it's the real, confirmed decode interface, not a guess.
- **Note for whoever owns the text-processing stage:** the real tokenization is
  `model._tokenize_texts([model._build_assistant_text(t) for t in texts])` -- i.e. it's not just a plain
  text tokenizer, there's a required prompt-building step (`_build_assistant_text`) before tokenization.
  If your teammate is building a separate/custom text tokenizer, it needs to replicate what
  `_build_assistant_text` does, or you keep using these two calls as-is for that stage.
